# Data Preprocessing — Explicit Exclusion Pipeline (rewrite)

**Design principles of this rewrite:**

1. **Mapping ≠ filtering.** `.map()` only translates codes; every row removal happens at a
   named, counted, explicit step.
2. **Every excluded record gets a reason.** Before any filtering, every origin/destination
   code is classified into one of a fixed set of categories. Counts are reported both as
   *rows* and as *persons* (migrant counts), since persons are what the methodology
   chapter must report.
3. **Whitelist, not blacklist.** The retained set is defined positively (both endpoints
   resolve to a 2011 London MSOA with a wealth decile). Special codes are *labelled* for
   the audit table but never used as the filtering mechanism — so an unanticipated
   pseudo-code can never leak in.
4. **Computational logic preserved.** IMD percentile normalisation, population-weighted
   MSOA aggregation, decile construction, cascade / counter-cascade formulas, and the
   four-flow consistency check are byte-for-byte the same as the previous notebook
   (`data_process_add_counter_cascades_20260610.ipynb`), so outputs are directly comparable.

**Pipeline order** (note: IMD/deciles now come *before* flow filtering, because the
analysis whitelist is defined as "MSOAs with a wealth decile"):

| Section | Step |
|---|---|
| 1 | Load all raw datasets (no modification) |
| 2 | Build harmonisation lookups (MSOA 2021→2011; OA→MSOA) |
| 3 | IMD percentiles, population-weighted MSOA IMD, wealth deciles → analysis whitelist |
| 4 | Classify every O-D endpoint code + exclusion audit tables (nothing dropped yet) |
| 5 | Explicit filtering with per-reason waterfall counts |
| 6 | Flow construction, cascade & counter-cascade features, consistency checks |
| 7 | Merge, exclusion-summary export, final feature export |


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pyprojroot import here
from scipy import stats
import matplotlib.colors as mcolors

sns.set_theme(style='whitegrid', font_scale=1.1)

ROOT = here()
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

# ---- File paths ----
imd_2010_path       = DATA_DIR / 'imd_2010.xls'
imd_2019_path       = DATA_DIR / 'imd_2019.csv'
census_od_2021_path = DATA_DIR / 'census_od_2021_msoa.csv'
census_od_2011_path = DATA_DIR / 'census_od_2011_oa.csv'
lookup_path         = DATA_DIR / 'NSPCL_NOV22_UK_LU.csv'
msoa_lookup_path    = DATA_DIR / 'msoa_2011_to_2021_lookup.csv'

---
## 1. Load All Datasets

Raw frames are loaded and **never modified in place**. All cleaning happens on copies at explicit, named steps. 

(The "20260610" notebook's early `'OD'` special-code drop is gone. Those rows are now classified in Section 4 and removed in Section 5, with counts.)

In [2]:
# ---- IMD ----
imd_2019 = pd.read_csv(imd_2019_path)
imd_2019.columns = imd_2019.columns.str.strip()

imd_2010 = pd.read_excel(imd_2010_path, sheet_name='IMD 2010')
imd_2010.columns = imd_2010.columns.str.strip()

In [3]:
# ---- Census O-D 2021 (MSOA level, 2021 codes) ----
ORIGIN_COL_2021 = 'Migrant MSOA one year ago code'
DEST_COL_2021   = 'Middle layer Super Output Areas code'
census_od_2021_raw = pd.read_csv(census_od_2021_path)
census_od_2021_raw[ORIGIN_COL_2021] = census_od_2021_raw[ORIGIN_COL_2021].astype(str).str.strip()
census_od_2021_raw[DEST_COL_2021]   = census_od_2021_raw[DEST_COL_2021].astype(str).str.strip()

# Detect 2021 count column
count_cols_2021 = [c for c in census_od_2021_raw.columns
                   if 'observation' in c.lower() or 'count' in c.lower()]
COUNT_COL_2021 = count_cols_2021[0] if count_cols_2021 else '_count'
if COUNT_COL_2021 == '_count':
    census_od_2021_raw[COUNT_COL_2021] = 1
print(f'2021 count column: {COUNT_COL_2021!r}')

2021 count column: 'Count'


In [7]:
# ---- Census O-D 2011 (OA level, 2011 codes) ----
OD_2011_ORIGIN_COL = 'origin_oa'
OD_2011_DEST_COL   = 'dest_oa'
OD_2011_COUNT_COL  = 'persons'
census_od_2011_raw = pd.read_csv(
    census_od_2011_path, header=None,
    names=[OD_2011_DEST_COL, OD_2011_ORIGIN_COL, OD_2011_COUNT_COL],   # A=dest, B=origin, C=count
    dtype={OD_2011_DEST_COL: str, OD_2011_ORIGIN_COL: str, OD_2011_COUNT_COL: int}
)
census_od_2011_raw[OD_2011_ORIGIN_COL] = census_od_2011_raw[OD_2011_ORIGIN_COL].str.strip()
census_od_2011_raw[OD_2011_DEST_COL]   = census_od_2011_raw[OD_2011_DEST_COL].str.strip()


In [8]:
# ---- Lookups ----
lookup = pd.read_csv(lookup_path, encoding='ISO-8859-1', low_memory=False)
msoa_11_21 = pd.read_csv(msoa_lookup_path)
msoa_11_21.columns = msoa_11_21.columns.str.strip().str.lower()

for name, df in [('IMD 2010', imd_2010), ('IMD 2019', imd_2019),
                 ('Census O-D 2021 (MSOA)', census_od_2021_raw),
                 ('Census O-D 2011 (OA)', census_od_2011_raw),
                 ('MSOA 2011-2021 Lookup', msoa_11_21)]:
    print(f'{name:28s} shape={df.shape}')

IMD 2010                     shape=(32482, 7)
IMD 2019                     shape=(32844, 57)
Census O-D 2021 (MSOA)       shape=(1490726, 5)
Census O-D 2011 (OA)         shape=(3709939, 3)
MSOA 2011-2021 Lookup        shape=(7201, 9)


---
## 2. Harmonisation Lookups

Two translation dictionaries. **These translate codes; they do not remove rows.**

- `msoa21_to_11`: 2021 MSOA → 2011 MSOA, restricted to *unchanged* (1:1) MSOAs.
  Boundary-changed (split/merged) MSOAs are deliberately absent — they will be
  classified as `boundary_changed` in the audit and excluded with a count.
- `oa_to_msoa_dict`: 2011 OA → 2011 MSOA (from the postcode lookup).

In [9]:
# ---- 2a. MSOA 2021 → 2011 (unchanged 1:1 only) ----
unchanged = msoa_11_21[msoa_11_21['msoa11cd'] == msoa_11_21['msoa21cd']].copy()
msoa21_to_11 = dict(zip(unchanged['msoa21cd'], unchanged['msoa11cd']))

# Full set of *real* 2021 MSOA codes (changed or not) — used by the classifier
# to detect "real MSOA can't be harmonised" or "not a georaphy code"
all_msoa21_codes = set(msoa_11_21['msoa21cd'].astype(str))

print(f'MSOA lookup rows:                  {len(msoa_11_21):,}')
print(f'Unchanged 1:1 (harmonisable):      {len(msoa21_to_11):,}')
print(f'Distinct 2021 codes (any status):  {len(all_msoa21_codes):,}')

MSOA lookup rows:                  7,201
Unchanged 1:1 (harmonisable):      7,080
Distinct 2021 codes (any status):  7,182


In [10]:
# ---- 2b. OA → MSOA ----
oa_to_msoa = lookup[['oa11cd', 'msoa11cd']].drop_duplicates().dropna()
oa_dup = oa_to_msoa.groupby('oa11cd')['msoa11cd'].nunique()
assert (oa_dup == 1).all(), f'{(oa_dup > 1).sum()} OAs map to multiple MSOAs!'
oa_to_msoa_dict = dict(zip(oa_to_msoa['oa11cd'], oa_to_msoa['msoa11cd']))
print(f'OA→MSOA mappings:                  {len(oa_to_msoa_dict):,}')

OA→MSOA mappings:                  232,044


---
## 3. IMD Percentiles, MSOA Aggregation, Wealth Deciles → Analysis Whitelist

Identical logic to the previous notebook. The output `wealth_dict` (keys = London MSOAs
with both IMD years and a 2010-based decile) **is the analysis whitelist**: a flow is
retained iff *both* endpoints are keys of `wealth_dict`.

In [11]:
# ---- 3a. London scope ----
london_boroughs = [
    'City of London', 'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent',
    'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich',
    'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering',
    'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea',
    'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham',
    'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton',
    'Tower Hamlets', 'Waltham Forest', 'Wandsworth', 'Westminster'
]
london_lookup = (lookup[lookup['ladnm'].isin(london_boroughs)]
                 [['lsoa11cd', 'msoa11cd', 'ladnm']].drop_duplicates())
london_msoas = set(london_lookup['msoa11cd'].unique())
print(f'London MSOAs (geography): {len(london_msoas)}')

London MSOAs (geography): 983


In [12]:
# ---- 3b. Rank → percentile normalisation ----
IMD_2010_LSOA_COL  = 'LSOA CODE'
IMD_2010_SCORE_COL = 'IMD SCORE'
IMD_2010_RANK_COL  = 'RANK OF IMD SCORE (where 1 is most deprived)'
IMD_2019_LSOA_COL  = 'LSOA code (2011)'
IMD_2019_SCORE_COL = 'Index of Multiple Deprivation (IMD) Score'
IMD_2019_RANK_COL  = 'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)'
POP_COL            = 'Total population: mid 2015 (excluding prisoners)'

n_lsoa_2010, n_lsoa_2019 = len(imd_2010), len(imd_2019)
imd_2010['IMD_Percentile_2010'] = imd_2010[IMD_2010_RANK_COL] / n_lsoa_2010
imd_2019['IMD_Percentile_2019'] = imd_2019[IMD_2019_RANK_COL] / n_lsoa_2019

In [ ]:
# ---- 3c. Population-weighted MSOA aggregation ----
lsoa_pop = (imd_2019[[IMD_2019_LSOA_COL, POP_COL]]
            .rename(columns={IMD_2019_LSOA_COL: 'lsoa11cd', POP_COL: 'pop'}).copy())

def pop_weighted_mean(df, value_col, weight_col='pop'):
    w = df[weight_col]
    if w.sum() == 0:
        return df[value_col].mean()
    return np.average(df[value_col], weights=w)

imd_2010_london = pd.merge(
    imd_2010[[IMD_2010_LSOA_COL, IMD_2010_SCORE_COL, IMD_2010_RANK_COL, 'IMD_Percentile_2010']],
    london_lookup, left_on=IMD_2010_LSOA_COL, right_on='lsoa11cd')
imd_2010_london = pd.merge(imd_2010_london, lsoa_pop, on='lsoa11cd', how='left')
imd_2010_london['pop'] = imd_2010_london['pop'].fillna(0)
msoa_imd_2010 = (imd_2010_london.groupby(['msoa11cd', 'ladnm'])
    .apply(lambda g: pd.Series({
        'IMD_2010':        pop_weighted_mean(g, IMD_2010_SCORE_COL),
        'IMD_Rank_2010':   pop_weighted_mean(g, IMD_2010_RANK_COL),
        'IMD_Pctile_2010': pop_weighted_mean(g, 'IMD_Percentile_2010')}))
    .reset_index())

imd_2019_london = pd.merge(
    imd_2019[[IMD_2019_LSOA_COL, IMD_2019_SCORE_COL, IMD_2019_RANK_COL,
              'IMD_Percentile_2019', POP_COL]],
    london_lookup, left_on=IMD_2019_LSOA_COL, right_on='lsoa11cd')
imd_2019_london = imd_2019_london.rename(columns={POP_COL: 'pop'})
imd_2019_london['pop'] = imd_2019_london['pop'].fillna(0)
msoa_imd_2019 = (imd_2019_london.groupby('msoa11cd')
    .apply(lambda g: pd.Series({
        'IMD_2019':        pop_weighted_mean(g, IMD_2019_SCORE_COL),
        'IMD_Rank_2019':   pop_weighted_mean(g, IMD_2019_RANK_COL),
        'IMD_Pctile_2019': pop_weighted_mean(g, 'IMD_Percentile_2019')}))
    .reset_index())

In [ ]:
# ---- 3d. Combine, change metrics, deciles ----
msoa_wealth = pd.merge(msoa_imd_2010, msoa_imd_2019, on='msoa11cd', how='inner')
msoa_wealth['IMD_Pctile_Change'] = msoa_wealth['IMD_Pctile_2019'] - msoa_wealth['IMD_Pctile_2010']
msoa_wealth['IMD_Score_Change']  = msoa_wealth['IMD_2019'] - msoa_wealth['IMD_2010']

msoa_wealth['Wealth_Decile'] = pd.qcut(msoa_wealth['IMD_2010'], 10, labels=False) + 1
msoa_wealth['Wealth_Decile'] = 11 - msoa_wealth['Wealth_Decile']        # 1=most deprived
msoa_wealth['Wealth_Decile_2019'] = pd.qcut(msoa_wealth['IMD_2019'], 10, labels=False) + 1
msoa_wealth['Wealth_Decile_2019'] = 11 - msoa_wealth['Wealth_Decile_2019']

wealth_dict      = msoa_wealth.set_index('msoa11cd')['Wealth_Decile'].to_dict()
wealth_dict_2019 = msoa_wealth.set_index('msoa11cd')['Wealth_Decile_2019'].to_dict()

# ---- THE ANALYSIS WHITELIST ----
analysis_msoas = set(wealth_dict.keys())
print(f'London MSOAs with both IMD years (= analysis whitelist): {len(analysis_msoas)}')
dropped_geo = london_msoas - analysis_msoas
if dropped_geo:
    print(f'NOTE: {len(dropped_geo)} London MSOAs lack an IMD decile and are excluded:')
    print(sorted(dropped_geo))

---
## 4. Endpoint Code Classification & Exclusion Audit

Every origin and destination code is labelled with **exactly one** category. Categories
are mutually exclusive and exhaustive, checked in priority order:

| Category | Meaning | Fate |
|---|---|---|
| `london_analysis` | Harmonises to a 2011 MSOA in the analysis whitelist | **retained** (if both ends) |
| `ew_non_london` | Harmonises to a valid 2011 MSOA outside London | excluded — out of scope |
| `boundary_changed` | Real 2021 MSOA, but split/merged ⇒ no 1:1 2011 code | excluded — unharmonisable |
| `scotland` / `n_ireland` | Real UK geography outside England & Wales | excluded — outside E&W frame |
| `special_code` | ONS pseudo-codes: `-8`, `999999999`, `N99…`, `OD…` (2011) | excluded — not a geography |
| `unknown` | Anything else (should be ~0; investigate if not) | excluded — flagged |

The classification is **for the audit table only** — the actual filter in Section 5 is
the positive whitelist test, so an unanticipated code can never slip through.

> Note: confirm the precise meaning of `-8` and `999999999` against the ONS variable
> documentation for this table (e.g. *does not apply* vs *origin not coded*), and cite
> it in the methodology chapter.

In [ ]:
SPECIAL_2021 = {'-8', '999999999'}

def classify_code_2021(code):
    """Classify a raw 2021 O-D endpoint code. Priority-ordered, exhaustive."""
    c = str(code).strip()
    if c in SPECIAL_2021 or c.startswith('N99'):
        return 'special_code'
    if c.startswith('S'):
        return 'scotland'
    if c.startswith('N'):
        return 'n_ireland'
    if c in msoa21_to_11:
        return 'london_analysis' if msoa21_to_11[c] in analysis_msoas else 'ew_non_london'
    if c in all_msoa21_codes:
        return 'boundary_changed'
    return 'unknown'

def classify_code_2011_oa(code):
    """Classify a raw 2011 OA-level endpoint code."""
    c = str(code).strip()
    if c.startswith('OD'):
        return 'special_code'          # cross-border / no-fixed-origin pseudo-OAs
    if c in oa_to_msoa_dict:
        return ('london_analysis' if oa_to_msoa_dict[c] in analysis_msoas
                else 'ew_non_london')
    if c.startswith('S'):
        return 'scotland'
    if c.startswith('N'):
        return 'n_ireland'
    return 'unknown'

CATEGORY_ORDER = ['london_analysis', 'ew_non_london', 'boundary_changed',
                  'scotland', 'n_ireland', 'special_code', 'unknown']

In [ ]:
def audit_endpoints(df, origin_col, dest_col, count_col, classifier, label):
    """
    Classify both endpoints of every flow record and tabulate, in rows AND persons.
    Returns (df_with_categories, audit_table). Drops nothing.
    """
    out = df.copy()
    # Classify unique codes once (fast), then map
    uniq = pd.unique(pd.concat([out[origin_col], out[dest_col]]).astype(str))
    cat_map = {c: classifier(c) for c in uniq}
    out['origin_cat'] = out[origin_col].astype(str).map(cat_map)
    out['dest_cat']   = out[dest_col].astype(str).map(cat_map)

    rows_tab = pd.crosstab(out['origin_cat'], out['dest_cat']) \
                 .reindex(index=CATEGORY_ORDER, columns=CATEGORY_ORDER, fill_value=0)
    persons_tab = pd.crosstab(out['origin_cat'], out['dest_cat'],
                              values=out[count_col], aggfunc='sum') \
                    .reindex(index=CATEGORY_ORDER, columns=CATEGORY_ORDER) \
                    .fillna(0).astype(int)

    print(f'\n================ {label}: endpoint audit ================')
    print(f'Total records: {len(out):,}   Total persons: {out[count_col].sum():,.0f}')
    print('\n--- Rows (origin category × dest category) ---')
    print(rows_tab.to_string())
    print('\n--- Persons (origin category × dest category) ---')
    print(persons_tab.to_string())

    n_unknown = (out['origin_cat'].eq('unknown') | out['dest_cat'].eq('unknown')).sum()
    if n_unknown:
        print(f'\n*** WARNING: {n_unknown:,} records contain UNKNOWN codes — investigate: ***')
        bad = out.loc[out['origin_cat'].eq('unknown'), origin_col].astype(str).value_counts().head(10)
        print(bad.to_string())
    return out, persons_tab

census_od_2021_aud, audit_2021 = audit_endpoints(
    census_od_2021_raw, ORIGIN_COL_2021, DEST_COL_2021, COUNT_COL_2021,
    classify_code_2021, '2021 O-D (MSOA)')

census_od_2011_aud, audit_2011 = audit_endpoints(
    census_od_2011_raw, OD_2011_ORIGIN_COL, OD_2011_DEST_COL, OD_2011_COUNT_COL,
    classify_code_2011_oa, '2011 O-D (OA)')

# Persist audit tables for the methodology chapter
audit_2021.to_csv(OUTPUT_DIR / 'exclusion_audit_2021.csv')
audit_2011.to_csv(OUTPUT_DIR / 'exclusion_audit_2011.csv')

---
## 5. Explicit Filtering (whitelist + waterfall)

A single rule, applied identically to both years:

> **Retain a flow record iff BOTH endpoints harmonise to a 2011 MSOA in the analysis
> whitelist (London, IMD decile available), AND origin ≠ destination at MSOA level.**

Each filtering step reports rows and persons removed, producing a *waterfall* you can
paste straight into the methodology chapter. The old standalone `!= '-8'` drop is gone:
`-8` is just one labelled species of non-whitelisted origin.

In [ ]:
def filter_to_london_flows(df, origin_col, dest_col, count_col,
                           code_to_msoa11, label):
    """
    Harmonise endpoints (translate only), then apply the whitelist filter with
    a counted waterfall. Returns the retained London-to-London flow table with
    origin_msoa11 / dest_msoa11 columns.
    """
    out = df.copy()
    n0, p0 = len(out), out[count_col].sum()

    # --- Harmonise (translation only; adds columns, removes nothing) ---
    out['origin_msoa11'] = out[origin_col].astype(str).map(code_to_msoa11)
    out['dest_msoa11']   = out[dest_col].astype(str).map(code_to_msoa11)

    # --- Whitelist masks (positive definition of the retained set) ---
    ok_origin = out['origin_msoa11'].isin(analysis_msoas)
    ok_dest   = out['dest_msoa11'].isin(analysis_msoas)

    steps = []
    def log(name, mask_keep):
        nonlocal out
        removed_rows    = (~mask_keep).sum()
        removed_persons = out.loc[~mask_keep, count_col].sum()
        out = out[mask_keep].copy()
        steps.append((name, removed_rows, removed_persons, len(out), out[count_col].sum()))

    log('origin not in London analysis set', ok_origin)
    ok_dest = out['dest_msoa11'].isin(analysis_msoas)          # recompute on survivor frame
    log('dest not in London analysis set', ok_dest)
    log('intra-MSOA (origin == dest)', out['origin_msoa11'] != out['dest_msoa11'])

    print(f'\n================ {label}: filtering waterfall ================')
    print(f'{"step":42s} {"rows -":>12s} {"persons -":>12s} {"rows left":>12s} {"persons left":>14s}')
    print(f'{"(start)":42s} {"":>12s} {"":>12s} {n0:>12,} {p0:>14,.0f}')
    for name, rr, rp, nl, pl in steps:
        print(f'{name:42s} {rr:>12,} {rp:>12,.0f} {nl:>12,} {pl:>14,.0f}')
    return out

# ---- 2021: codes are MSOA-2021 → translate via msoa21_to_11 ----
london_od_2021 = filter_to_london_flows(
    census_od_2021_aud, ORIGIN_COL_2021, DEST_COL_2021, COUNT_COL_2021,
    msoa21_to_11, '2021 O-D')

# ---- 2011: codes are OA-2011 → translate via oa_to_msoa_dict, then aggregate ----
london_od_2011_oa = filter_to_london_flows(
    census_od_2011_aud, OD_2011_ORIGIN_COL, OD_2011_DEST_COL, OD_2011_COUNT_COL,
    oa_to_msoa_dict, '2011 O-D (OA level)')

census_od_2011_msoa = (london_od_2011_oa
    .groupby(['origin_msoa11', 'dest_msoa11'])[OD_2011_COUNT_COL].sum()
    .reset_index().rename(columns={OD_2011_COUNT_COL: 'count'}))
print(f'\n2011 aggregated MSOA-to-MSOA flow records: {len(census_od_2011_msoa):,}')
print(f'2011 total London-internal migrants:        {census_od_2011_msoa["count"].sum():,.0f}')

> **Note on the 2011 intra-MSOA step.** The OA-level frame is filtered *before*
> aggregation, and `origin_msoa11 != dest_msoa11` at OA level already removes
> within-MSOA moves, so no second intra-MSOA filter is needed after the groupby —
> but the assert in Section 6 will catch it if anything slips.

---
## 6. Flow Construction & Cascade Features

`build_flows` now **asserts** its input is already London-whitelisted — it no longer
performs any silent filtering. The decile maps cannot introduce NaNs because every
surviving endpoint is, by construction, a key of `wealth_dict`. Cascade,
counter-cascade and consistency-check logic are unchanged from the previous notebook.

In [ ]:
def build_flows(od_df, origin_col, dest_col, count_col, wealth_mapping, year_label):
    """Attach deciles and summarise. Input MUST already be whitelist-filtered."""
    df = od_df.copy()
    df['Origin_Decile'] = df[origin_col].map(wealth_mapping)
    df['Dest_Decile']   = df[dest_col].map(wealth_mapping)

    n_nan = df[['Origin_Decile', 'Dest_Decile']].isna().any(axis=1).sum()
    assert n_nan == 0, (
        f'{n_nan} rows lack a decile — input was not properly whitelist-filtered. '
        f'No silent dropping is permitted in this function.')

    df['Origin_Decile'] = df['Origin_Decile'].astype(int)
    df['Dest_Decile']   = df['Dest_Decile'].astype(int)
    df['Decile_Shift']  = df['Dest_Decile'] - df['Origin_Decile']

    flow_matrix = df.pivot_table(index='Origin_Decile', columns='Dest_Decile',
                                 values=count_col, aggfunc='sum', fill_value=0)
    total    = df[count_col].sum()
    upward   = df.loc[df['Decile_Shift'] > 0, count_col].sum()
    downward = df.loc[df['Decile_Shift'] < 0, count_col].sum()
    lateral  = df.loc[df['Decile_Shift'] == 0, count_col].sum()
    flow_direction = pd.Series({'Upward': upward, 'Downward': downward,
                                'Lateral': lateral, 'Total': total})
    shift_dist = df.groupby('Decile_Shift')[count_col].sum()

    print(f'\n=== {year_label} Flow Summary ===')
    print(f'  London-to-London records: {len(df):,}')
    print(f'  Total migrants:           {total:,.0f}')
    print(f'  Upward:  {upward:>10,.0f} ({upward/total*100:.1f}%)')
    print(f'  Down:    {downward:>10,.0f} ({downward/total*100:.1f}%)')
    print(f'  Lateral: {lateral:>10,.0f} ({lateral/total*100:.1f}%)')
    return df, flow_matrix, flow_direction, shift_dist

In [ ]:
def compute_base_flows(london_flow, count_col, origin_msoa_col, dest_msoa_col):
    """Per-MSOA base flow counts (unchanged logic)."""
    inflow_w = (london_flow[london_flow['Origin_Decile'] > london_flow['Dest_Decile']]
                .groupby(dest_msoa_col)[count_col].sum().rename('Inflow_Wealthier'))
    outflow_p = (london_flow[london_flow['Dest_Decile'] < london_flow['Origin_Decile']]
                 .groupby(origin_msoa_col)[count_col].sum().rename('Outflow_Poorer'))
    total_in  = london_flow.groupby(dest_msoa_col)[count_col].sum().rename('Total_Inflow')
    total_out = london_flow.groupby(origin_msoa_col)[count_col].sum().rename('Total_Outflow')

    base_flows = pd.DataFrame(index=inflow_w.index.union(outflow_p.index)
                              .union(total_in.index).union(total_out.index))
    base_flows.index.name = 'msoa11cd'
    for s in [inflow_w, outflow_p, total_in, total_out]:
        base_flows = base_flows.join(s, how='left')
    return base_flows.fillna(0)


def compute_cascade_features(base_flows):
    """Derived cascade metrics (unchanged logic)."""
    cascade = base_flows.copy()
    cascade['Total_Migration'] = cascade['Total_Inflow'] + cascade['Total_Outflow']
    cascade['CFI_Churn'] = cascade['Inflow_Wealthier'] + cascade['Outflow_Poorer']
    cascade['CFI_Rate'] = np.where(
        cascade['Total_Migration'] > 0,
        (cascade['Inflow_Wealthier'] * cascade['Outflow_Poorer']) / cascade['Total_Migration'], 0)
    cascade['Net_Cascade'] = cascade['Inflow_Wealthier'] - cascade['Outflow_Poorer']
    cascade['Pct_Inflow_Wealthier'] = np.where(
        cascade['Total_Inflow'] > 0,
        (cascade['Inflow_Wealthier'] / cascade['Total_Inflow']) * 100, 0)
    return cascade


def compute_counter_flows(london_flow, count_col, origin_msoa_col, dest_msoa_col):
    """Counter-cascade base flows (unchanged logic)."""
    outflow_w = (london_flow[london_flow['Dest_Decile'] > london_flow['Origin_Decile']]
                 .groupby(origin_msoa_col)[count_col].sum().rename('Outflow_Wealthier'))
    inflow_p = (london_flow[london_flow['Origin_Decile'] < london_flow['Dest_Decile']]
                .groupby(dest_msoa_col)[count_col].sum().rename('Inflow_Poorer'))
    total_in  = london_flow.groupby(dest_msoa_col)[count_col].sum().rename('Total_Inflow')
    total_out = london_flow.groupby(origin_msoa_col)[count_col].sum().rename('Total_Outflow')

    counter_flows = pd.DataFrame(index=outflow_w.index.union(inflow_p.index)
                                 .union(total_in.index).union(total_out.index))
    counter_flows.index.name = 'msoa11cd'
    for s in [outflow_w, inflow_p, total_in, total_out]:
        counter_flows = counter_flows.join(s, how='left')
    return counter_flows.fillna(0)


def compute_counter_cascade_features(counter_flows):
    """Counter-cascade metrics (unchanged logic)."""
    cc = counter_flows.copy()
    cc['Total_Migration'] = cc['Total_Inflow'] + cc['Total_Outflow']
    cc['Counter_Churn'] = cc['Outflow_Wealthier'] + cc['Inflow_Poorer']
    cc['Counter_Rate'] = np.where(
        cc['Total_Migration'] > 0,
        (cc['Outflow_Wealthier'] * cc['Inflow_Poorer']) / cc['Total_Migration'], 0)
    cc['Net_Counter'] = cc['Outflow_Wealthier'] - cc['Inflow_Poorer']
    cc['Pct_Outflow_Wealthier'] = np.where(
        cc['Total_Outflow'] > 0,
        (cc['Outflow_Wealthier'] / cc['Total_Outflow']) * 100, 0)
    return cc


def verify_four_flows(cascade_df, counter_df, year_label):
    """Cascade + counter-cascade + lateral must reconstruct totals (unchanged logic)."""
    check = pd.DataFrame(index=cascade_df.index)
    check['IW'] = cascade_df['Inflow_Wealthier']
    check['IP'] = counter_df['Inflow_Poorer']
    check['TI'] = cascade_df['Total_Inflow']
    check['Inflow_Lateral'] = check['TI'] - check['IW'] - check['IP']
    check['OP'] = cascade_df['Outflow_Poorer']
    check['OW'] = counter_df['Outflow_Wealthier']
    check['TO'] = cascade_df['Total_Outflow']
    check['Outflow_Lateral'] = check['TO'] - check['OP'] - check['OW']
    in_neg  = (check['Inflow_Lateral']  < -0.01).sum()
    out_neg = (check['Outflow_Lateral'] < -0.01).sum()
    print(f'\n=== {year_label} Four-Flow Consistency Check ===')
    print(f'  Negative inflow-lateral residuals:  {in_neg} / {len(check)}')
    print(f'  Negative outflow-lateral residuals: {out_neg} / {len(check)}')
    status = '✓ consistent' if in_neg == 0 and out_neg == 0 else '✗ INCONSISTENT'
    print(f'  {status}')
    return check

In [ ]:
# ---- Run the pipeline for both years (+ IMD-2019-decile robustness variant) ----
london_flow_2011, flow_matrix_2011, flow_dir_2011, shift_dist_2011 = build_flows(
    census_od_2011_msoa, 'origin_msoa11', 'dest_msoa11', 'count', wealth_dict, '2011')
base_flows_2011 = compute_base_flows(london_flow_2011, 'count', 'origin_msoa11', 'dest_msoa11')
cascade_2011 = compute_cascade_features(base_flows_2011)

london_flow_2021, flow_matrix_2021, flow_dir_2021, shift_dist_2021 = build_flows(
    london_od_2021, 'origin_msoa11', 'dest_msoa11', COUNT_COL_2021, wealth_dict, '2021')
base_flows_2021 = compute_base_flows(london_flow_2021, COUNT_COL_2021, 'origin_msoa11', 'dest_msoa11')
cascade_2021 = compute_cascade_features(base_flows_2021)

london_flow_2021_alt, *_ = build_flows(
    london_od_2021, 'origin_msoa11', 'dest_msoa11', COUNT_COL_2021,
    wealth_dict_2019, '2021 (IMD 2019 deciles)')
base_flows_2021_alt = compute_base_flows(london_flow_2021_alt, COUNT_COL_2021,
                                         'origin_msoa11', 'dest_msoa11')
cascade_2021_alt = compute_cascade_features(base_flows_2021_alt)

# Counter-cascades
counter_2011 = compute_counter_cascade_features(
    compute_counter_flows(london_flow_2011, 'count', 'origin_msoa11', 'dest_msoa11'))
counter_2021 = compute_counter_cascade_features(
    compute_counter_flows(london_flow_2021, COUNT_COL_2021, 'origin_msoa11', 'dest_msoa11'))

check_2011 = verify_four_flows(cascade_2011, counter_2011, '2011')
check_2021 = verify_four_flows(cascade_2021, counter_2021, '2021')

---
## 7. Merge, Exclusion Summary & Export

Two exports:
1. **`msoa_cascade_features_<date>.csv`** — the analysis dataset (same schema as before).
2. **`exclusion_summary.csv`** — one row per (year × exclusion reason) with rows and
   persons removed: the table for the methodology chapter.

In [ ]:
# ---- 7a. Merge cascade + counter-cascade features ----
msoa_analysis = msoa_wealth[['msoa11cd', 'ladnm', 'Wealth_Decile', 'Wealth_Decile_2019',
                             'IMD_2010', 'IMD_2019', 'IMD_Rank_2010', 'IMD_Rank_2019',
                             'IMD_Pctile_2010', 'IMD_Pctile_2019',
                             'IMD_Pctile_Change', 'IMD_Score_Change']].copy()

c11  = cascade_2011.add_suffix('_11')
c21  = cascade_2021.add_suffix('_21')
cc11 = counter_2011[['Outflow_Wealthier', 'Inflow_Poorer', 'Counter_Churn',
                     'Counter_Rate', 'Net_Counter', 'Pct_Outflow_Wealthier']].add_suffix('_11')
cc21 = counter_2021[['Outflow_Wealthier', 'Inflow_Poorer', 'Counter_Churn',
                     'Counter_Rate', 'Net_Counter', 'Pct_Outflow_Wealthier']].add_suffix('_21')

msoa_analysis = (msoa_analysis
    .merge(c11,  left_on='msoa11cd', right_index=True, how='left')
    .merge(c21,  left_on='msoa11cd', right_index=True, how='left')
    .merge(cc11, left_on='msoa11cd', right_index=True, how='left')
    .merge(cc21, left_on='msoa11cd', right_index=True, how='left')
    .merge(cascade_2021_alt[['Net_Cascade', 'Inflow_Wealthier', 'Outflow_Poorer']]
           .rename(columns={'Net_Cascade': 'Net_Cascade_21_alt',
                            'Inflow_Wealthier': 'Inflow_Wealthier_21_alt',
                            'Outflow_Poorer': 'Outflow_Poorer_21_alt'}),
           left_on='msoa11cd', right_index=True, how='left'))

# MSOAs with zero retained flows in a year legitimately get 0, not NaN
flow_cols = [c for c in msoa_analysis.columns if c.endswith(('_11', '_21', '_21_alt'))]
msoa_analysis[flow_cols] = msoa_analysis[flow_cols].fillna(0)

print(f'Final analysis dataset: {msoa_analysis.shape[0]} MSOAs × {msoa_analysis.shape[1]} columns')

In [ ]:
# ---- 7b. Build the methodology exclusion summary from the audit tables ----
def exclusion_summary(persons_tab, year):
    """Collapse the origin×dest category matrix into per-reason exclusions.
    Priority: a record is attributed to the origin's reason first, then dest's."""
    rows = []
    total = persons_tab.values.sum()
    retained = persons_tab.loc['london_analysis', 'london_analysis']
    rows.append({'year': year, 'reason': 'RETAINED (London→London, pre intra-MSOA filter)',
                 'persons': int(retained), 'pct_of_total': retained / total * 100})
    for cat in CATEGORY_ORDER:
        if cat == 'london_analysis':
            continue
        # excluded because of origin in this category (any dest)
        p_o = persons_tab.loc[cat, :].sum()
        # excluded because of dest in this category, but origin was fine
        p_d = persons_tab.loc['london_analysis', cat]
        p = p_o + p_d
        if p > 0:
            rows.append({'year': year, 'reason': f'excluded: {cat}',
                         'persons': int(p), 'pct_of_total': p / total * 100})
    return pd.DataFrame(rows)

excl = pd.concat([exclusion_summary(audit_2011, 2011),
                  exclusion_summary(audit_2021, 2021)], ignore_index=True)
excl['pct_of_total'] = excl['pct_of_total'].round(2)
print(excl.to_string(index=False))
excl.to_csv(OUTPUT_DIR / 'exclusion_summary.csv', index=False)

In [ ]:
# ---- 7c. Export ----
from datetime import date
out_path = OUTPUT_DIR / f'msoa_cascade_features_{date.today():%Y%m%d}.csv'
msoa_analysis.to_csv(out_path, index=False)
print(f'Exported: {out_path}')
print(f'Also exported: exclusion_summary.csv, exclusion_audit_2011.csv, exclusion_audit_2021.csv')

---
## Validation against the previous pipeline

Because Sections 3 and 6 reuse the old logic verbatim and the whitelist filter retains
exactly the rows the old `dropna` retained, the feature columns should **match the
previous export** (e.g. `msoa_cascade_features_20260522.csv`) to numerical precision.
Run this check once after the first execution:

```python
old = pd.read_csv('msoa_cascade_features_20260522.csv')
new = msoa_analysis
shared = [c for c in old.columns if c in new.columns]
m = old[shared].sort_values('msoa11cd').reset_index(drop=True) \
       .compare(new[shared].sort_values('msoa11cd').reset_index(drop=True))
print('Differences:' , m.shape)   # expect (0, 0)
```

If differences appear, the audit tables tell you exactly which exclusion category
diverged — which is the entire point of this rewrite.